we will be learning about Bagging and boosting in this section

In [3]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

sns.set_style("whitegrid")
plt.style.use("fivethirtyeight")

Loading the dataset

In [4]:
df =pd.read_csv('diabetes.csv')

In [5]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [7]:
#checking for null data points in the data

df.isnull().sum()

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

There are no null values in the dataset

In [8]:
df.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


In [9]:
categorical_val = []
continous_val = []
for column in df.columns:
#     print('==============================')
#     print(f"{column} : {df[column].unique()}")
    if len(df[column].unique()) <= 10:
        categorical_val.append(column)
    else:
        continous_val.append(column)

In [10]:
continous_val

['Pregnancies',
 'Glucose',
 'BloodPressure',
 'SkinThickness',
 'Insulin',
 'BMI',
 'DiabetesPedigreeFunction',
 'Age']

Data-Precessing

In [11]:
df.columns

Index(['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin',
       'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'],
      dtype='str')

In [12]:
# How many missing zeros are mising in each feature
feature_columns = [
    'Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 
    'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age'
]

for column in feature_columns:
    print("============================================")
    print(f"{column} ==> Missing zeros : {len(df.loc[df[column] == 0])}")

Pregnancies ==> Missing zeros : 111
Glucose ==> Missing zeros : 5
BloodPressure ==> Missing zeros : 35
SkinThickness ==> Missing zeros : 227
Insulin ==> Missing zeros : 374
BMI ==> Missing zeros : 11
DiabetesPedigreeFunction ==> Missing zeros : 0
Age ==> Missing zeros : 0


### What is a "Missing Zero"?

In biological/medical datasets like the Diabetes dataset, a **"missing zero"** (or *invalid zero*) refers to a **value recorded as `0` to represent missing, unrecorded, or unknown data** rather than a genuine physical measurement of zero.

#### Feature Validity Check for `0`:
- **Glucose, BloodPressure, SkinThickness, Insulin, BMI**: `0` is **physically impossible** for a living person. These `0`s are actually missing data points that require imputation.
- **Pregnancies**: `0` is a **valid** measurement (indicates zero past pregnancies).
- **Age, DiabetesPedigreeFunction**: Cannot be `0` (min age in dataset is 21).

#### Why it matters:
Leaving `0` as-is will distort feature statistics (like mean, std) and cause ML models to treat unrecorded entries as actual zero values.

In [13]:
# Define only the columns where 0 is physically IMPOSSIBLE (missing data)
invalid_zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 
    'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']

Mean Imputation

In [26]:

# 2. Replace 0 with NaN on df
df[invalid_zero_cols] = df[invalid_zero_cols].replace(0, np.nan)
# 3. Impute missing values on df (e.g., using mean)
for col in invalid_zero_cols:
    df[col] = df[col].fillna(df[col].mean())

In [27]:
# 4. NOW separate X and y (both will have 768 rows)
X = df.drop('Outcome', axis=1)
y = df['Outcome']
print(X.shape, y.shape)  # Should both be (768, 8) and (768,)

(768, 8) (768,)


In [28]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

In [29]:
# model evaluation

from sklearn.metrics import  confusion_matrix, classification_report, accuracy_score

def evaluate(model, X_train, X_test, y_train, y_test):
    y_test_pred = model.predict(X_test)
    y_train_pred = model.predict(X_train)

    print('Training results:\n ===============')

    clf_report = pd.DataFrame(classification_report(y_train,y_train_pred, output_dict=True))
    print(f"Confusion Matrix:\n {confusion_matrix(y_train, y_train_pred)}")
    print(f"accuracy score:\n{accuracy_score(y_train, y_train_pred)}")
    print(f"classifcation report:\n {clf_report}")

    print("Testing results: \n ================")
    clf_report = pd.DataFrame(classification_report(y_test,y_test_pred, output_dict=True))
    print(f"Confusion Matrix:\n {confusion_matrix(y_test,y_test_pred)}")
    print(f"accuracy score:\n{accuracy_score(y_test,y_test_pred)}")
    print(f"classifcation report:\n {clf_report}")



## Bagging Algorithms
Bootstrap Aggregation or bagging involves taking multiple samples from your training dataset (with replacement) and training a model for each sample.

The final output prediction is averaged across the predictions of all of the sub-models.

The three bagging models covered in this section are as follows:

1. Bagged Decision Trees
2. Random Forest
3. Extra Trees

## 1. Bagged Decision Trees
Bagging performs best with algorithms that have high variance. A popular example is decision trees, often constructed without pruning.

**BaggingClassifier**:

A Bagging classifier is an ensemble meta-estimator that fits base classifiers each on random subsets of the original dataset and then aggregates their individual predictions (either by voting or by averaging) to form a final prediction. Such a meta-estimator can typically be used as a way to reduce the variance of a black-box estimator (e.g., a decision tree), by introducing randomization into its construction procedure and then making an ensemble out of it.

This algorithm encompasses several works from the literature. When random subsets of the dataset are drawn as random subsets of the samples, then this algorithm is known as Pasting. If samples are drawn with replacement, then the method is known as Bagging. When random subsets of the dataset are drawn as random subsets of the features, then the method is known as Random Subspaces. Finally, when base estimators are built on subsets of both samples and features, then the method is known as Random Patches.

**BaggingClassifier Parameters:**
- `base_estimator`: The base estimator to fit on random subsets of the dataset. If None, then the base estimator is a decision tree.
***
- `n_estimators`: The number of base estimators in the ensemble.
***
- `max_samples`: The number of samples to draw from X to train each base estimator.
***
- `max_features`: The number of features to draw from X to train each base estimator.
***
- `bootstrap`: Whether samples are drawn with replacement. If False, sampling without replacement is performed.
***
- `bootstrap_features`: Whether features are drawn with replacement.
***
- `oob_score`: Whether to use out-of-bag samples to estimate the generalization error.
***
- `warm_start`: When set to True, reuse the solution of the previous call to fit and add more estimators to the ensemble, otherwise, just fit a whole new ensemble.

In [33]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import  DecisionTreeClassifier

tree = DecisionTreeClassifier()

bagging_clf= BaggingClassifier(n_estimators=300,random_state=42)

bagging_clf.fit(X_train, y_train)

evaluate(bagging_clf, X_train, X_test, y_train, y_test)

Training results:
Confusion Matrix:
 [[401   0]
 [  0 213]]
accuracy score:
1.0
classifcation report:
                0      1  accuracy  macro avg  weighted avg
precision    1.0    1.0       1.0        1.0           1.0
recall       1.0    1.0       1.0        1.0           1.0
f1-score     1.0    1.0       1.0        1.0           1.0
support    401.0  213.0       1.0      614.0         614.0
Testing results: 
Confusion Matrix:
 [[79 20]
 [16 39]]
accuracy score:
0.7662337662337663
classifcation report:
                    0          1  accuracy   macro avg  weighted avg
precision   0.831579   0.661017  0.766234    0.746298      0.770664
recall      0.797980   0.709091  0.766234    0.753535      0.766234
f1-score    0.814433   0.684211  0.766234    0.749322      0.767925
support    99.000000  55.000000  0.766234  154.000000    154.000000


estimatorobject, default=None
The base estimator to fit on random subsets of the dataset. If None, then the base estimator is a DecisionTreeClassifier.

## 2. Random Forest

A random forest is a meta-estimator that fits several decision tree classifiers on various sub-samples of the dataset and uses averaging to improve the predictive accuracy and control over-fitting.

The sub-sample size is always the same as the original input sample size but the samples are drawn with replacement if `bootstrap=True` (default).

- **Random forest algorithm parameters:**
- `n_estimators`: The number of trees in the forest.
*** 
- `criterion`: The function to measure the quality of a split. Supported criteria are "`gini`" for the Gini impurity and "`entropy`" for the information gain.
***
- `max_depth`: The maximum depth of the tree. If None, then nodes are expanded until all leaves are pure or until all leaves contain less than `min_samples_split` samples.
***
- `min_samples_split`: The minimum number of samples required to split an internal node.
***
- `min_samples_leaf`: The minimum number of samples required to be at a leaf node. A split point at any depth will only be considered if it leaves at least ``min_samples_leaf`` training samples in each of the left and right branches.  This may have the effect of smoothing the model, especially in regression.
***
- `min_weight_fraction_leaf`: The minimum weighted fraction of the total of weights (of all the input samples) required to be at a leaf node. Samples have equal weight when sample_weight is not provided.
***
- `max_features`: The number of features to consider when looking for the best split.
***
- `max_leaf_nodes`: Grow a tree with ``max_leaf_nodes`` in best-first fashion. Best nodes are defined as relative reduction in impurity. If None then an unlimited number of leaf nodes.
***
- `min_impurity_decrease`: A node will be split if this split induces a decrease of the impurity greater than or equal to this value.
***
- `min_impurity_split`: Threshold for early stopping in tree growth. A node will split if its impurity is above the threshold, otherwise, it is a leaf.
***
- `bootstrap`: Whether bootstrap samples are used when building trees. If False, the whole dataset is used to build each tree.
***
- `oob_score`: Whether to use out-of-bag samples to estimate the generalization accuracy.
***
- `warm_start`: When set to True, reuse the solution of the previous call to fit and add more estimators to the ensemble, otherwise, just fit a whole new ensemble.

In [35]:
from sklearn.ensemble import RandomForestClassifier

rf_clf = RandomForestClassifier(random_state=42, n_estimators=100)

rf_clf.fit(X_train,y_train)

evaluate(rf_clf,X_train, X_test,y_train, y_test)

Training results:
Confusion Matrix:
 [[401   0]
 [  0 213]]
accuracy score:
1.0
classifcation report:
                0      1  accuracy  macro avg  weighted avg
precision    1.0    1.0       1.0        1.0           1.0
recall       1.0    1.0       1.0        1.0           1.0
f1-score     1.0    1.0       1.0        1.0           1.0
support    401.0  213.0       1.0      614.0         614.0
Testing results: 
Confusion Matrix:
 [[80 19]
 [19 36]]
accuracy score:
0.7532467532467533
classifcation report:
                    0          1  accuracy   macro avg  weighted avg
precision   0.808081   0.654545  0.753247    0.731313      0.753247
recall      0.808081   0.654545  0.753247    0.731313      0.753247
f1-score    0.808081   0.654545  0.753247    0.731313      0.753247
support    99.000000  55.000000  0.753247  154.000000    154.000000


## 3. Extra Trees
Extra Trees are another modification of bagging where random trees are constructed from samples of the training dataset.

You can construct an Extra Trees model for classification using the ExtraTreesClassifier class.

**ExtraTreeClassifier**:

This class implements a meta-estimator that fits a number of randomized decision trees (a.k.a. extra-trees) on various sub-samples of the dataset and uses averaging to improve the predictive accuracy and control over-fitting.

**ExtraTreeClassifier Parameters**:
- `n_estimators`: The number of trees in the forest.
*** 
- `criterion`: The function to measure the quality of a split. Supported criteria are "`gini`" for the Gini impurity and "`entropy`" for the information gain.
***
- `max_depth`: The maximum depth of the tree. If None, then nodes are expanded until all leaves are pure or until all leaves contain less than `min_samples_split` samples.
***
- `min_samples_split`: The minimum number of samples required to split an internal node.
***
- `min_samples_leaf`: The minimum number of samples required to be at a leaf node. A split point at any depth will only be considered if it leaves at least ``min_samples_leaf`` training samples in each of the left and right branches.  This may have the effect of smoothing the model, especially in regression.
***
- `min_weight_fraction_leaf`: The minimum weighted fraction of the sum total of weights (of all the input samples) required to be at a leaf node. Samples have equal weight when sample_weight is not provided.
***
- `max_features`: The number of features to consider when looking for the best split.
***
- `max_leaf_nodes`: Grow a tree with ``max_leaf_nodes`` in best-first fashion. Best nodes are defined as relative reduction in impurity. If None then an unlimited number of leaf nodes.
***
- `min_impurity_decrease`: A node will be split if this split induces a decrease of the impurity greater than or equal to this value.
***
- `min_impurity_split`: Threshold for early stopping in tree growth. A node will split if its impurity is above the threshold, otherwise, it is a leaf.
***
- `bootstrap`: Whether bootstrap samples are used when building trees. If False, the whole dataset is used to build each tree.
***
- `oob_score`: Whether to use out-of-bag samples to estimate the generalization accuracy.
***
- `warm_start`: When set to True, reuse the solution of the previous call to fit and add more estimators to the ensemble, otherwise, just fit a whole new ensemble.

In [36]:
from sklearn.ensemble import ExtraTreesClassifier

ex_tree_clf = ExtraTreesClassifier(n_estimators=100, max_features =7, random_state=42)

ex_tree_clf.fit(X_train,y_train)

evaluate(ex_tree_clf, X_train,X_test, y_train,y_test)

Training results:
Confusion Matrix:
 [[401   0]
 [  0 213]]
accuracy score:
1.0
classifcation report:
                0      1  accuracy  macro avg  weighted avg
precision    1.0    1.0       1.0        1.0           1.0
recall       1.0    1.0       1.0        1.0           1.0
f1-score     1.0    1.0       1.0        1.0           1.0
support    401.0  213.0       1.0      614.0         614.0
Testing results: 
Confusion Matrix:
 [[78 21]
 [15 40]]
accuracy score:
0.7662337662337663
classifcation report:
                    0          1  accuracy   macro avg  weighted avg
precision   0.838710   0.655738  0.766234    0.747224      0.773363
recall      0.787879   0.727273  0.766234    0.757576      0.766234
f1-score    0.812500   0.689655  0.766234    0.751078      0.768627
support    99.000000  55.000000  0.766234  154.000000    154.000000


# Boosting Algorithms
Boosting ensemble algorithms creates a sequence of models that attempt to correct the mistakes of the models before them in the sequence.

Once created, the models make predictions that may be weighted by their demonstrated accuracy and the results are combined to create a final output prediction.

The two most common boosting ensemble machine learning algorithms are:

1. AdaBoost
2. Stochastic Gradient Boosting
***

## 1. AdaBoost
AdaBoost was perhaps the first successful boosting ensemble algorithm. It generally works by weighting instances in the dataset by how easy or difficult they are to classify, allowing the algorithm to pay less attention to them in the construction of subsequent models.

You can construct an AdaBoost model for classification using the AdaBoostClassifier class.

**AdaBoostClassifier**:

An AdaBoost classifier is a meta-estimator that begins by fitting a classifier on the original dataset and then fits additional copies of the classifier on the same dataset but where the weights of incorrectly classified instances are adjusted such that subsequent classifiers focus more on difficult cases.

**AdaBoostClassifier Params**:
- `base_estimator`: The base estimator from which the boosted ensemble is built.
***
- `n_estimators`: The maximum number of estimators at which boosting is terminated. In case of the perfect fit, the learning procedure is stopped early.
***
- `learning_rate`: The learning rate shrinks the contribution of each classifier by ``learning_rate``. There is a trade-off between ``learning_rate`` and ``n_estimators``.
***
- `algorithm`: If 'SAMME.R' then use the SAMME.R real boosting algorithm. ``base_estimator`` must support the calculation of class probabilities. If 'SAMME' then use the SAMME discrete boosting algorithm. The SAMME.R algorithm typically converges faster than SAMME, achieving a lower test error with fewer boosting iterations.

In [38]:
from sklearn.ensemble import  AdaBoostClassifier

ada_boost_clf = AdaBoostClassifier(n_estimators=51)

ada_boost_clf.fit(X_train,y_train)

evaluate(ada_boost_clf, X_train, X_test,y_train,y_test)

Training results:
Confusion Matrix:
 [[352  49]
 [ 71 142]]
accuracy score:
0.8045602605863192
classifcation report:
                     0           1  accuracy   macro avg  weighted avg
precision    0.832151    0.743455   0.80456    0.787803      0.801382
recall       0.877805    0.666667   0.80456    0.772236      0.804560
f1-score     0.854369    0.702970   0.80456    0.778670      0.801848
support    401.000000  213.000000   0.80456  614.000000    614.000000
Testing results: 
Confusion Matrix:
 [[78 21]
 [16 39]]
accuracy score:
0.7597402597402597
classifcation report:
                    0          1  accuracy   macro avg  weighted avg
precision   0.829787   0.650000   0.75974    0.739894      0.765578
recall      0.787879   0.709091   0.75974    0.748485      0.759740
f1-score    0.808290   0.678261   0.75974    0.743276      0.761851
support    99.000000  55.000000   0.75974  154.000000    154.000000


## 2. Stochastic Gradient Boosting
Stochastic Gradient Boosting (also called Gradient Boosting Machines) is one of the most sophisticated ensemble techniques. It is also a technique that is proving to be perhaps of the best techniques available for improving performance via ensembles.

**GradientBoostingClassifier**:

GB builds an additive model in a forward stage-wise fashion; it allows for the optimization of arbitrary differentiable loss functions. In each stage ``n_classes_`` regression trees are fit on the negative gradient of the binomial or multinomial deviance loss function. Binary classification is a special case where only a single regression tree is induced.

**GradientBoostingClassifier Parameters**:

- `loss`: loss function to be optimized. 'deviance' refers to deviance (= logistic regression) for classification with probabilistic outputs. For loss 'exponential' gradient boosting recovers the AdaBoost algorithm.
***
- `learning_rate`: learning rate shrinks the contribution of each tree by `learning_rate`. There is a trade-off between learning_rate and n_estimators.
***
- `n_estimators`: The number of boosting stages to perform. Gradient boosting is fairly robust to over-fitting so a large number usually results in better performance.
***
- `subsample`: The fraction of samples to be used for fitting the individual base learners. If smaller than 1.0 this results in Stochastic Gradient Boosting. `subsample` interacts with the parameter `n_estimators`. Choosing `subsample < 1.0` leads to a reduction of variance and an increase in bias.
***
- `criterion`: The function to measure the quality of a split. Supported criteria are "friedman_mse" for the mean squared error with an improvement score by Friedman, "`mse`" for the mean squared error, and "`mae`" for the mean absolute error. The default value of "friedman_mse" is generally the best as it can provide a better approximation in some cases.
***
- `min_samples_split`: The minimum number of samples required to split an internal node.
***
- `min_samples_leaf`: The minimum number of samples required to be at a leaf node. A split point at any depth will only be considered if it leaves at least ``min_samples_leaf`` training samples in each of the left and right branches.  This may have the effect of smoothing the model, especially in regression.
***
- `min_weight_fraction_leaf`: The minimum weighted fraction of the sum total of weights (of all the input samples) required to be at a leaf node. Samples have equal weight when sample_weight is not provided.
***
- `max_depth`: maximum depth of the individual regression estimators. The maximum depth limits the number of nodes in the tree. Tune this parameter for best performance; the best value depends on the interaction of the input variables.
***
- `min_impurity_decrease`: A node will be split if this split induces a decrease of the impurity greater than or equal to this value.
***
- `min_impurity_split`: Threshold for early stopping in tree growth. A node will split if its impurity is above the threshold, otherwise, it is a leaf.
***
- `max_features`: The number of features to consider when looking for the best split.
***
- `max_leaf_nodes`: Grow trees with ``max_leaf_nodes`` in best-first fashion. Best nodes are defined as relative reduction in impurity. If None then an unlimited number of leaf nodes.
***
- `warm_start`: When set to ``True``, reuse the solution of the previous call to fit and add more estimators to the ensemble, otherwise, just erase the previous solution.
***
- `validation_fraction`: The proportion of training data to set aside as validation set for early stopping. Must be between 0 and 1. Only used if ``n_iter_no_change`` is set to an integer.
***
- `n_iter_no_change`: used to decide if early stopping will be used to terminate training when the validation score is not improving. By default, it is set to None to disable early stopping. If set to a number, it will set aside the ``validation_fraction`` size of the training data as validation and terminate training when the validation score is not improving in all of the previous ``n_iter_no_change`` numbers of iterations. The split is stratified.
***
- `tol`: Tolerance for the early stopping. When the loss is not improving by at least tol for ``n_iter_no_change`` iterations (if set to a number), the training stops.
***
- `ccp_alpha`: Complexity parameter used for Minimal Cost-Complexity Pruning. The subtree with the largest cost complexity that is smaller than ``ccp_alpha`` will be chosen.
***

In [40]:
from sklearn.ensemble import  GradientBoostingClassifier

grad_boost_clf = GradientBoostingClassifier(n_estimators=100, random_state=42)
grad_boost_clf.fit(X_train,y_train)
evaluate(grad_boost_clf,X_train,X_test,y_train,y_test)

Training results:
Confusion Matrix:
 [[389  12]
 [ 32 181]]
accuracy score:
0.9283387622149837
classifcation report:
                     0           1  accuracy   macro avg  weighted avg
precision    0.923990    0.937824  0.928339    0.930907      0.928789
recall       0.970075    0.849765  0.928339    0.909920      0.928339
f1-score     0.946472    0.891626  0.928339    0.919049      0.927445
support    401.000000  213.000000  0.928339  614.000000    614.000000
Testing results: 
Confusion Matrix:
 [[75 24]
 [17 38]]
accuracy score:
0.7337662337662337
classifcation report:
                    0          1  accuracy   macro avg  weighted avg
precision   0.815217   0.612903  0.733766    0.714060      0.742962
recall      0.757576   0.690909  0.733766    0.724242      0.733766
f1-score    0.785340   0.649573  0.733766    0.717456      0.736852
support    99.000000  55.000000  0.733766  154.000000    154.000000


# Voting Ensemble

Voting is one of the simplest ways of combining predictions from multiple machine learning algorithms.

It works by first creating two or more standalone models from your training dataset. A Voting Classifier can then be used to wrap your models and average the predictions of the sub-models when asked to make predictions for new data.

The predictions of the sub-models can be weighted, but specifying the weights for classifiers manually or even heuristically is difficult. More advanced methods can learn how to best weight the predictions from submodels, but this is called stacking (stacked generalization) and is currently not provided in Scikit-learn.

**VotingClassifier** : 
- `estimators`: Invoking the ``fit`` method on the ``VotingClassifier`` will fit clones of those original estimators that will be stored in the class attribute ``self.estimators_``.
***
- `voting`: If 'hard', uses predicted class labels for majority rule voting. Else if 'soft', predict the class label based on the argmax of the sums of the predicted probabilities, which is recommended for an ensemble of well-calibrated classifiers.
***

In [41]:
from sklearn.ensemble import  VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

estimators =[]

log_reg = LogisticRegression(solver='liblinear')
estimators.append(('Logistic', log_reg))

tree = DecisionTreeClassifier()
estimators.append(('Tree', tree))

svm_clf = SVC(gamma='scale')
estimators.append(('SVM', svm_clf))

voting= VotingClassifier(estimators=estimators)

voting.fit(X_train, y_train)

evaluate(voting, X_train,X_test, y_train, y_test)


Training results:
Confusion Matrix:
 [[375  26]
 [ 93 120]]
accuracy score:
0.8061889250814332
classifcation report:
                     0           1  accuracy   macro avg  weighted avg
precision    0.801282    0.821918  0.806189    0.811600      0.808441
recall       0.935162    0.563380  0.806189    0.749271      0.806189
f1-score     0.863061    0.668524  0.806189    0.765792      0.795575
support    401.000000  213.000000  0.806189  614.000000    614.000000
Testing results: 
Confusion Matrix:
 [[89 10]
 [23 32]]
accuracy score:
0.7857142857142857
classifcation report:
                    0          1  accuracy   macro avg  weighted avg
precision   0.794643   0.761905  0.785714    0.778274      0.782951
recall      0.898990   0.581818  0.785714    0.740404      0.785714
f1-score    0.843602   0.659794  0.785714    0.751698      0.777956
support    99.000000  55.000000  0.785714  154.000000    154.000000


Reference: https://www.kaggle.com/code/faressayah/ensemble-ml-algorithms-bagging-boosting-voting/notebook